# Task 1 Train Notebook (`google/gemma-4-E4B-it` on 2x T4)

Notebook nay duoc viet rieng cho setup `2x T4`.

Nguyen tac:
- text-only SFT
- 4-bit + LoRA
- max_length ngan
- smoke test truoc
- chi train that khi smoke qua


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REPO_DIR = Path.cwd()
print("Repo dir:", REPO_DIR)
os.chdir(REPO_DIR)
%pip install -U pip setuptools wheel
%pip install -r requirements.txt
%pip install --no-cache-dir bitsandbytes==0.48.2
%pip install -U git+https://github.com/huggingface/transformers.git


In [ ]:
import torch, transformers, huggingface_hub, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("peft:", peft.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv,noheader


In [ ]:
from huggingface_hub import login

HF_TOKEN = "YOUR_HF_TOKEN"
MODEL_ID = "google/gemma-4-E4B-it"
RUN_NAME = "task1_gemma4_e4b_2xt4"
HF_MODEL_REPO_ID = "SpringWang08/multimodal-empathy-mental-health-gemma4-e4b-2xt4"

login(HF_TOKEN, add_to_git_credential=False)
print("HF login ok")


In [ ]:
from transformers import AutoProcessor, AutoTokenizer

processor = None
try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    print("Processor loaded:", processor.__class__.__name__)
except Exception as e:
    print("AutoProcessor unavailable in current environment:", repr(e))
    print("Falling back to tokenizer-only path for text-only SFT.")

tokenizer = getattr(processor, "tokenizer", None)
if tokenizer is None:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded:", tokenizer.__class__.__name__)
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)


In [ ]:
!bash scripts/download_avamerg.sh
!bash scripts/download_esconv.sh
!mkdir -p outputs/sft outputs/eval

from pathlib import Path
ava_path = Path("data/raw/avamerg/train.json")
esc_path = Path("data/raw/esconv/ESConv.json")
print("AvaMERG file exists:", ava_path.exists(), ava_path)
print("ESConv file exists:", esc_path.exists(), esc_path)
if not ava_path.exists() or not esc_path.exists():
    raise FileNotFoundError("Dataset download chua xong hoac bi fail.")


In [ ]:
from argparse import Namespace
from scripts.train_sft import run_training

base_cfg = dict(
    model_name_or_path=MODEL_ID,
    avamerg_root="data/raw/avamerg",
    avamerg_split="train",
    avamerg_text_only=True,
    esconv_json="data/raw/esconv/ESConv.json",
    output_dir="outputs/sft/debug_gemma4_e4b_2xt4",
    max_length=256,
    max_response_tokens=64,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    num_train_epochs=1.0,
    logging_steps=5,
    save_steps=100,
    warmup_ratio=0.03,
    max_steps=-1,
    use_lora=True,
    load_in_4bit=True,
    lora_r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    report_to="none",
    dump_example_prompts=True,
    max_train_samples=8,
    gradient_checkpointing=True,
)

run_training(Namespace(**base_cfg))


In [ ]:
!sed -n '1,200p' outputs/sft/debug_gemma4_e4b_2xt4/example_prompts.json


In [ ]:
smoke_cfg = dict(base_cfg)
smoke_cfg.update({
    "output_dir": "outputs/sft/gemma4_e4b_2xt4_smoke",
    "dump_example_prompts": False,
    "max_train_samples": 8,
    "max_length": 256,
    "max_response_tokens": 64,
    "gradient_accumulation_steps": 1,
    "max_steps": 1,
    "logging_steps": 1,
})

run_training(Namespace(**smoke_cfg))


In [ ]:
# Chi chay cell nay sau khi smoke test da qua.
train_cfg = dict(base_cfg)
train_cfg.update({
    "output_dir": f"outputs/sft/{RUN_NAME}",
    "dump_example_prompts": False,
    "max_train_samples": None,
    "max_length": 256,
    "max_response_tokens": 64,
    "gradient_accumulation_steps": 8,
    "gradient_checkpointing": True,
    "logging_steps": 10,
    "save_steps": 100,
    "max_steps": -1,
})

run_training(Namespace(**train_cfg))
